# AVD894 — UVFA Treasure Hunt

This notebook runs the implementation through Intermediate Task 2. It is an orchestration and presentation notebook: the maintained implementation remains in `src/` and `scripts/`.

Paper: Tom Schaul et al., *Universal Value Function Approximators*, ICML 2015.

The simulator is a course-project adaptation. Results are generated locally and are not copied from the paper.

In [8]:
from pathlib import Path
import json
import sys
import subprocess
import time

# This works whether the notebook is opened from the project root or notebooks/.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook inside New Paper Project')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

CONFIG_TASK1 = PROJECT_ROOT / 'configs' / 'task1_small.json'
CONFIG_TASK2 = PROJECT_ROOT / 'configs' / 'task2_default.json'
ARTIFACTS = PROJECT_ROOT / 'artifacts' / 'notebook_run'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Artifacts:', ARTIFACTS)

Project: d:\Acad. PDFs\SEM 7\Reinforcement Learning\New Paper Project
Artifacts: d:\Acad. PDFs\SEM 7\Reinforcement Learning\New Paper Project\artifacts\notebook_run


## 1. Runtime check

The project automatically uses CUDA when PyTorch reports it as available and otherwise falls back to CPU.

In [9]:
import numpy as np
import torch

print('Python executable:', sys.executable)
print('NumPy:', np.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Python executable: c:\Users\T RAJESH\AppData\Local\Programs\Python\Python312\python.exe
NumPy: 2.4.6
PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 2. Intermediate Task 1 — exact value iteration

This cell solves the finite state-goal MDP and stores the same JSON/CSV artifacts as the command-line script.

In [10]:
from goal_treasure.exact import follow_policy, value_iteration
from goal_treasure.utils import load_json, make_env, save_json

task1_config = load_json(CONFIG_TASK1)
task1_env = make_env(task1_config)
task1_result = value_iteration(
    task1_env.obstacles,
    gamma=task1_config['gamma'],
    step_reward=task1_config['step_reward'],
    wall_penalty=task1_config['wall_penalty'],
    goal_reward=task1_config['goal_reward'],
    tolerance=task1_config['tolerance'],
    max_iterations=task1_config['max_iterations'],
)
start, goal = task1_env.free_cells[0], task1_env.free_cells[-1]
task1_path = follow_policy(task1_result.policy, start, goal, task1_env.obstacles)
task1_summary = {
    'state_goal_pairs': len(task1_result.values),
    'free_cells': len(task1_env.free_cells),
    'actions': task1_env.action_size,
    'iterations': task1_result.iterations,
    'final_delta': task1_result.delta,
    'example_path_length': len(task1_path) - 1,
}
save_json(ARTIFACTS / 'task1_summary.json', task1_summary)
print(json.dumps(task1_summary, indent=2))
print('Path:', task1_path)

{
  "state_goal_pairs": 462,
  "free_cells": 22,
  "actions": 4,
  "iterations": 10,
  "final_delta": 0.0,
  "example_path_length": 9
}
Path: [(1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6)]


In [11]:
# Simple notebook visualization for Task 1's exact path.
from IPython.display import display, Markdown

task1_env.reset(start=start, goal=goal)
display(Markdown('```text\n' + task1_env.render_ascii(task1_path) + '\n```'))

```text
########
#A.....#
#·####.#
#·.....#
#·####.#
#·····G#
########
```

## 3. Intermediate Task 2 — train from this notebook

The default below is the submission run. For a quick smoke check, change both episode counts to 20 before using Run All.

In [12]:
TASK2_EPISODES = 1000
TASK2_OUTPUT = ARTIFACTS / f'training_{TASK2_EPISODES}_episodes'

train_command = [
    sys.executable, str(PROJECT_ROOT / 'scripts' / 'train_task2.py'),
    '--config', str(CONFIG_TASK2),
    '--episodes', str(TASK2_EPISODES),
    '--output', str(TASK2_OUTPUT),
    '--device', 'auto',
]
started = time.perf_counter()
train_run = subprocess.run(train_command, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(train_run.stdout)
if train_run.returncode != 0:
    print(train_run.stderr)
    raise RuntimeError(f'Training failed with exit code {train_run.returncode}')
print(f'Training wall time: {time.perf_counter() - started:.1f} seconds')
CHECKPOINT = TASK2_OUTPUT / 'checkpoint.pt'
print('Checkpoint:', CHECKPOINT)

episode=100/1000 success=0.620 loss=0.1539 device=cuda
episode=200/1000 success=0.860 loss=0.2015 device=cuda
episode=300/1000 success=0.850 loss=0.1729 device=cuda
episode=400/1000 success=0.900 loss=0.1264 device=cuda
episode=500/1000 success=1.000 loss=0.1387 device=cuda
episode=600/1000 success=1.000 loss=0.1506 device=cuda
episode=700/1000 success=0.970 loss=0.1246 device=cuda
episode=800/1000 success=0.990 loss=0.1069 device=cuda
episode=900/1000 success=1.000 loss=0.1012 device=cuda
episode=1000/1000 success=1.000 loss=0.0899 device=cuda
Saved checkpoint to d:\Acad. PDFs\SEM 7\Reinforcement Learning\New Paper Project\artifacts\notebook_run\training_1000_episodes\checkpoint.pt

Training wall time: 54.5 seconds
Checkpoint: d:\Acad. PDFs\SEM 7\Reinforcement Learning\New Paper Project\artifacts\notebook_run\training_1000_episodes\checkpoint.pt


## 4. Evaluate on new random test episodes

The evaluation seeds are separate from the training seeds. The shortest-path controller is a reference baseline, not the RL agent.

In [13]:
EVAL_EPISODES = 500
EVALUATION = TASK2_OUTPUT / 'evaluation.json'
eval_command = [
    sys.executable, str(PROJECT_ROOT / 'scripts' / 'evaluate_task2.py'),
    '--config', str(CONFIG_TASK2),
    '--checkpoint', str(CHECKPOINT),
    '--episodes', str(EVAL_EPISODES),
    '--output', str(EVALUATION),
    '--device', 'auto',
]
eval_run = subprocess.run(eval_command, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(eval_run.stdout)
if eval_run.returncode != 0:
    print(eval_run.stderr)
    raise RuntimeError(f'Evaluation failed with exit code {eval_run.returncode}')
evaluation = json.loads(EVALUATION.read_text(encoding='utf-8'))
display(evaluation['learned'])
display(evaluation['shortest_path_reference'])
print('Evaluation file:', EVALUATION)

learned success rate: 0.986
shortest-path reference success rate: 1.000
saved evaluation to d:\Acad. PDFs\SEM 7\Reinforcement Learning\New Paper Project\artifacts\notebook_run\training_1000_episodes\evaluation.json



{'success_rate': 0.986,
 'mean_steps_successful': 4.545638945233265,
 'mean_return': 9.901200000000001}

{'success_rate': 1.0, 'mean_steps': 4.28, 'mean_return': 10.138200000000001}

Evaluation file: d:\Acad. PDFs\SEM 7\Reinforcement Learning\New Paper Project\artifacts\notebook_run\training_1000_episodes\evaluation.json


## 5. Display one learned rollout

This cell displays a new random episode as text. For the full clickable window, run `scripts/demo.py` once from the terminal or VS Code task.

In [14]:
from goal_treasure.uvfa import load_checkpoint
from goal_treasure.utils import device_from_request, make_env

demo_config = load_json(CONFIG_TASK2)
demo_env = make_env(demo_config)
device = device_from_request('auto')
network, metadata = load_checkpoint(CHECKPOINT, device)
demo_env.reset(seed=123456)
path = [demo_env.position]
total_reward = 0.0
for _ in range(demo_env.max_steps):
    with torch.no_grad():
        q_values = network(
            torch.from_numpy(demo_env.state_vector()).unsqueeze(0).to(device),
            torch.from_numpy(demo_env.goal_vector()).unsqueeze(0).to(device),
        )
    action = int(q_values.argmax(dim=1).item())
    result = demo_env.step(action)
    path.append(demo_env.position)
    total_reward += result.reward
    if result.terminated or result.truncated:
        break
print({'success': demo_env.position == demo_env.goal, 'steps': demo_env.steps, 'return': total_reward})
display(Markdown('```text\n' + demo_env.render_ascii(path) + '\n```'))

{'success': True, 'steps': 5, 'return': 10.2}


```text
########
#...·..#
#..#·..#
#..#·..#
#...··A#
#...#..#
#......#
########
```

## 6. What to submit

Use the generated JSON files as evidence, and use `docs/TASK1_REPORT.md`, `docs/TASK2_REPORT.md`, and `docs/RESULTS.md` when preparing the Moodle report. Do not call the smoke-run metrics paper results. The notebook output can be exported to HTML or PDF from Jupyter for presentation.

## Extended training comparison (measured run)
This compares the existing 1,000-episode baseline and 5,000-episode run on the same held-out seeds. Training and test JSON files are local artifacts and are excluded from Git; the values below are also written in docs/RESULTS.md.


In [15]:
import json, math
from pathlib import Path
import numpy as np

baseline_dir = ARTIFACTS / 'task2_submission_1000'
extended_dir = ARTIFACTS / 'task2_extended_5000'
if not (baseline_dir / 'evaluation.json').exists() or not (extended_dir / 'evaluation.json').exists():
    print('Run the Task 2 training/evaluation commands in README.md to generate both artifact folders.')
else:
    bootstrap_rng = np.random.default_rng(2026)
    for label, folder in [('1,000 episodes', baseline_dir), ('5,000 episodes', extended_dir)]:
        result = json.loads((folder / 'evaluation.json').read_text(encoding='utf-8'))
        rows = result['learned_episode_rows']
        successes = sum(row['success'] for row in rows)
        n = len(rows); rate = successes / n; z = 1.96
        denom = 1 + z*z/n
        center = (rate + z*z/(2*n)) / denom
        half = z * math.sqrt(rate*(1-rate)/n + z*z/(4*n*n)) / denom
        steps = np.array([row['steps'] for row in rows if row['success']], dtype=float)
        draws = bootstrap_rng.choice(steps, size=(20000, len(steps)), replace=True).mean(axis=1)
        lo, hi = np.quantile(draws, [0.025, 0.975])
        print(f'{label}: {successes}/{n} success; rate={rate:.1%}, Wilson 95% CI [{center-half:.2%}, {center+half:.2%}]; successful mean steps={steps.mean():.3f}, bootstrap 95% CI [{lo:.3f}, {hi:.3f}]; mean return={result["learned"]["mean_return"]:.4f}')


1,000 episodes: 493/500 success; rate=98.6%, Wilson 95% CI [97.14%, 99.32%]; successful mean steps=4.546, bootstrap 95% CI [4.349, 4.744]; mean return=9.9012
5,000 episodes: 500/500 success; rate=100.0%, Wilson 95% CI [99.24%, 100.00%]; successful mean steps=4.284, bootstrap 95% CI [4.104, 4.466]; mean return=10.1378


## Launch the interactive demo window
Run this cell to open the Tkinter game window using the 5,000-episode checkpoint. The window is separate from VS Code; use Auto-play for a live rollout or Next RL step to explain decisions one at a time.


In [ ]:
from pathlib import Path
import subprocess, sys

GUI_CHECKPOINT = ARTIFACTS / 'task2_extended_5000' / 'checkpoint.pt'
if not GUI_CHECKPOINT.exists():
    GUI_CHECKPOINT = ARTIFACTS / 'task2_submission_1000' / 'checkpoint.pt'
if not GUI_CHECKPOINT.exists():
    GUI_CHECKPOINT = TASK2_OUTPUT / 'checkpoint.pt'
if not GUI_CHECKPOINT.exists():
    raise FileNotFoundError('Run the Task 2 training cell first; no checkpoint was found.')
GUI_PROCESS = subprocess.Popen(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'demo.py'),
     '--config', str(CONFIG_TASK2), '--checkpoint', str(GUI_CHECKPOINT)],
    cwd=PROJECT_ROOT,
)
print(f'Opened Goal-Conditioned Treasure Hunt using {GUI_CHECKPOINT}')
